In [1]:
import opensim as osim
import math

In [16]:
# Define the model
model = osim.Model()
model.setName('torque_example')
model.setGravity(osim.Vec3(0, -9.81, 0))
ground = model.getGround()

# Define the body
m = 1.0 
J = osim.Inertia(0.1, 0.1, 0.1)
pos_com = osim.Vec3(0, 0, 0)

head = osim.Body()
head.setName('head')
head.setMass(m)
head.setInertia(J)
head.setMassCenter(pos_com)
model.addBody(head)

# Pin joint for rotation of the head
joint_name = 'head_joint'
parent_physical_frame = ground
location_in_parent = osim.Vec3(0, 1, 0)
orientation_in_parent = osim.Vec3(math.pi/2, 0, 0)
child_physical_frame = head
location_in_child = osim.Vec3(0, 0, 0)
orientation_in_child = osim.Vec3(0, 0, 0)

joint = osim.PinJoint(
    joint_name,
    parent_physical_frame,
    location_in_parent,
    orientation_in_parent,
    child_physical_frame,
    location_in_child,
    orientation_in_child
)

model.addJoint(joint)

# Define coordinate
coord = joint.updCoordinate()
coord.setName('q')
coord.setDefaultValue(0)
coord.setDefaultSpeedValue(0)

print(f'Joint name: {joint.getName()}')
print(f'Coordinate name: {coord.getName()}')

Joint name: head_joint
Coordinate name: q


In [17]:
# Define CoordinateActuator
act = osim.CoordinateActuator()
act.setName('torque')
act.setCoordinate(coord)
act.setOptimalForce(1)
act.setMinControl(-1e6)
act.setMaxControl(1e6)
model.addForce(act)

# Define controller
func = osim.Constant()
ctrl = osim.PrescribedController()
ctrl.setName('controller')
ctrl.addActuator(act)
ctrl.prescribeControlForActuator('torque', func)
model.addController(ctrl)

In [33]:
# Finalize and initialize the model
model.finalizeConnections()
state = model.initSystem()

model.realizePosition(state)
model.realizeVelocity(state)
model.realizeAcceleration(state)
model.realizeDynamics(state)

print(f'Initial position:       {coord.getValue(state)} rad')
print(f'Initial speed:          {coord.getSpeedValue(state)} rad/s')
print(f'Initial acceleration:   {coord.getAccelerationValue(state)} rad/s²')
print(f'Initial torque:         {act.getActuation(state)} Nm')

Initial position:       0.0 rad
Initial speed:          0.0 rad/s
Initial acceleration:   0.0 rad/s²
Initial torque:         0.0 Nm


In [34]:
# Enable overriding actuator
act.overrideActuation(state, True)
act.setOverrideActuation(state, 5.0)

model.realizeDynamics(state)
model.realizeAcceleration(state)

print(f'Overridden torque:       {act.getActuation(state)} Nm')
print(f'Overridden acceleration: {coord.getAccelerationValue(state)} rad/s²')

Overridden torque:       5.0 Nm
Overridden acceleration: 50.0 rad/s²


In [35]:
# Initialize Simulation Manager
manager = osim.Manager(model)
manager.setIntegratorAccuracy(1e-6)
manager.initialize(state)

In [36]:
# Define simulation parameters
t = 0.0
dt = 0.01
tf = 0.5

In [37]:
# Perform one simulation step
state = manager.integrate(t + dt)
model.realizePosition(state)
model.realizeVelocity(state)
model.realizeAcceleration(state)
model.realizeDynamics(state)

t = state.getTime()

print(f'After {t} seconds:')
print(f'Position:       {coord.getValue(state)} rad')
print(f'Speed:          {coord.getSpeedValue(state)} rad/s')
print(f'Acceleration:   {coord.getAccelerationValue(state)} rad/s²')
print(f'Torque:         {act.getActuation(state)} Nm')

After 0.01 seconds:
Position:       0.0025 rad
Speed:          0.5 rad/s
Acceleration:   50.0 rad/s²
Torque:         5.0 Nm


In [31]:
# Initialize Simulation Manager
manager = osim.Manager(model)

In [39]:
# Apply a non-zero torque and simulate again
torque_val = 1.0
act.setActuation(state, torque_val)

model.realizeDynamics(state)
model.realizeAcceleration(state)


state = manager.integrate(t + dt)
model.realizePosition(state)
model.realizeVelocity(state)
model.realizeAcceleration(state)
model.realizeDynamics(state)

t = state.getTime()

print(f'After {t:.2f} seconds with torque applied:')
print(f'Position:       {coord.getValue(state):.4f} rad')
print(f'Speed:          {coord.getSpeedValue(state)} rad/s')
print(f'Acceleration:   {coord.getAccelerationValue(state)} rad/s²')
print(f'Torque:         {act.getActuation(state)} Nm')

After 0.02 seconds with torque applied:
Position:       0.0080 rad
Speed:          0.6 rad/s
Acceleration:   10.0 rad/s²
Torque:         1.0 Nm


In [25]:
# Apply a non-zero torque and simulate again
torque_val = 1.0
act.setActuation(state, torque_val)
model.realizeAcceleration(state)

state = manager.integrate(t + dt)
model.realizePosition(state)
model.realizeVelocity(state)
model.realizeAcceleration(state)
model.realizeDynamics(state)

t = state.getTime()

print(f'After {t:.2f} seconds with torque applied:')
print(f'Position:       {coord.getValue(state):.4f} rad')
print(f'Speed:          {coord.getSpeedValue(state)} rad/s')
print(f'Acceleration:   {coord.getAccelerationValue(state)} rad/s²')
print(f'Torque:         {act.getActuation(state)} Nm')

After 0.03 seconds with torque applied:
Position:       0.0225 rad
Speed:          1.5 rad/s
Acceleration:   50.0 rad/s²
Torque:         5.0 Nm
